# Unit 1

## Building Serverless APIs

Welcome back! In the previous lesson, you learned the theory of serverless computing on Google Cloud Platform (GCP). Now, you are ready to take the next step: defining, testing, and deploying your applications using GCP's professional configuration formats.

On GCP, you manage serverless resources by describing your application in configuration files, specifically **OpenAPI specifications**. This "Infrastructure as Code" approach allows you to define your API endpoints and link them to your Cloud Functions systematically.

### In this lesson, you will learn how to:
* Define your Cloud Function and API endpoint using OpenAPI.
* Test your function locally using the **Functions Framework**.
* Deploy your application to GCP and verify it works in production.

---

### Quick Recap: Cloud Functions and API Gateway

Before we dive in, remember the two-part harmony of GCP serverless:
1.  **Cloud Function:** Your logic. It processes input and returns a response.
2.  **API Gateway:** Your front door. It handles security and routes requests to the correct function.



---

### 1. Defining Your Application with OpenAPI

To connect API Gateway to a Cloud Function, you use an **OpenAPI specification** file (`openapi.yaml`). This describes your API in a standard format that GCP understands.

#### The Cloud Function (`main.py`)
This is the logic we will be deploying:
```python
import json

def calculate_tax(request):
    try:
        body = request.get_json(silent=True) or {}
        amount = float(body.get("amount", 0))
        tax = round(amount * 0.07, 2)
        total = round(amount + tax, 2)
        response = {"amount": amount, "tax": tax, "total": total}
        return (json.dumps(response), 200, {"Content-Type": "application/json"})
    except Exception as e:
        return (json.dumps({"error": str(e)}), 400, {"Content-Type": "application/json"})
```

#### The API Gateway Config (`openapi.yaml`)
This YAML file tells GCP where to find your function:
```yaml
openapi: 3.0.0
info:
  title: Tax Calculator API
  version: 1.0.0
paths:
  /calculate:
    post:
      summary: Calculate tax and total
      operationId: calculateTax
      x-google-backend:
        address: https://REGION-PROJECT_ID.cloudfunctions.net/calculate_tax
      responses:
        '200':
          description: Calculation result
```
> **Note:** The `x-google-backend` field is a special GCP extension that points the Gateway to your function's URL.

---

### 2. Testing Locally with Sample Data

Testing locally saves time and prevents "deployment fatigue." GCP provides the **Functions Framework** to run functions on your own machine.

#### Installation and Running
```bash
pip install functions-framework
functions-framework --target=calculate_tax
```

#### Sending a Test Request
In a separate terminal, use `curl` to simulate an API call:
```bash
curl -X POST "http://localhost:8080" \
     -H "Content-Type: application/json" \
     -d '{"amount": 150}'
```
**Expected Output:**
`{"amount": 150.0, "tax": 10.5, "total": 160.5}`

---

### 3. Deploying to GCP

Once local tests pass, it is time to go live. This involves a three-step process:

#### Step A: Deploy the Function
```bash
gcloud functions deploy calculate_tax \
  --entry-point calculate_tax \
  --runtime python310 \
  --trigger-http \
  --memory 256MB
```

#### Step B: Deploy the API Gateway
First, create the configuration, then create the gateway itself:
```bash
# Create the config
gcloud api-gateway api-configs create tax-cfg \
  --api=tax-api --openapi-spec=openapi.yaml

# Create the gateway
gcloud api-gateway gateways create tax-gateway \
  --api=tax-api --api-config=tax-cfg --location=REGION
```

#### Step C: Verify the Live Endpoint
Retrieve your URL and test the live production environment:
```bash
gcloud api-gateway gateways describe tax-gateway --location=REGION
```
Use the `defaultHostname` provided in the output to send a final `curl` request. If you see your JSON result, your serverless API is officially live!

---

### Summary and What's Next
You’ve just moved from writing simple code to architecting a cloud-deployed API. You learned how to:
1.  **Standardize** your API using OpenAPI.
2.  **Validate** logic locally with the Functions Framework.
3.  **Launch** production resources using `gcloud`.

You are now ready to practice these steps yourself in the upcoming exercises. Good luck, and have fun building!

## Fix the OpenAPI Configuration for Cloud Function Endpoint

You have been given an OpenAPI configuration file that is supposed to connect an API Gateway endpoint to a Google Cloud Function for calculating tax. However, there is a problem: the API Gateway is not set up correctly to handle POST requests to the /calculate endpoint.

Your job is to fix the openapi.yaml file so that the API responds properly to POST requests at the correct path and connects to the deployed Cloud Function.

Check the following in the OpenAPI spec:

    The HTTP method should be correct for receiving JSON data.
    The path should match what the Cloud Function expects.
    The backend address should point to the correct Cloud Function URL.

Once you fix the configuration, you can test your changes locally using the provided event data to make sure everything works before deployment. This exercise will help you understand how GCP connects Cloud Functions to API endpoints and prepare you for building your own serverless APIs.

```yaml
openapi: 3.0.0
info:
  title: Tax Calculator API
  version: 1.0.0
paths:
  /calc:
    get:
      summary: Calculate tax and total
      operationId: calculateTax
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                amount:
                  type: number
      responses:
        '200':
          description: Calculation result
          content:
            application/json:
              schema:
                type: object
                properties:
                  amount:
                    type: number
                  tax:
                    type: number
                  total:
                    type: number
      # TODO: Fix the backend address and method/path above
      x-google-backend:
        address: https://REGION-PROJECT_ID.cloudfunctions.net/calculate_tax

```

```python
import json

def calculate_tax(request):
    try:
        body = request.get_json(silent=True) or {}
        amount = float(body.get("amount", 0))
        tax = round(amount * 0.07, 2)
        total = round(amount + tax, 2)
        response = {
            "amount": amount,
            "tax": tax,
            "total": total
        }
        return (json.dumps(response), 200, {"Content-Type": "application/json"})
    except Exception as e:
        return (json.dumps({"error": str(e)}), 400, {"Content-Type": "application/json"})
```

```json
{
  "amount": 35
}
```

To fix the configuration, we need to align the OpenAPI specification with the behavior of your Cloud Function. 

There were three main issues in the original YAML:
1.  **The Method:** You were using `get`. In standard API design, `GET` requests shouldn't have a `requestBody`. Since we are sending JSON data, we must use `post`.
2.  **The Path:** The function expects the `/calculate` endpoint, but the YAML was using `/calc`.
3.  **Logical Consistency:** The `x-google-backend` is correctly pointing to the Cloud Function, but it will only trigger if the path and method above it are correct.



### Fixed `openapi.yaml`

```yaml
openapi: 3.0.0
info:
  title: Tax Calculator API
  version: 1.0.0
paths:
  # FIX: Changed path from /calc to /calculate
  /calculate:
    # FIX: Changed method from get to post to support requestBody
    post:
      summary: Calculate tax and total
      operationId: calculateTax
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                amount:
                  type: number
      responses:
        '200':
          description: Calculation result
          content:
            application/json:
              schema:
                type: object
                properties:
                  amount:
                    type: number
                  tax:
                    type: number
                  total:
                    type: number
      # The x-google-backend connects the Gateway to the actual Function
      x-google-backend:
        address: https://REGION-PROJECT_ID.cloudfunctions.net/calculate_tax
```

### Why these changes are critical:

* **POST vs GET:** While you *can* technically force some systems to accept a body on a GET request, it violates the HTTP specification. GCP's API Gateway follows these standards strictly; `POST` is the correct verb for "submitting" data for processing.
* **Routing Accuracy:** API Gateway acts like a map. If a user tries to reach `/calculate` but your map only has `/calc`, they will receive a `404 Not Found` error even if your Cloud Function is running perfectly.
* **The Backend Hook:** The `x-google-backend` is a "vendor extension." It’s how Google knows that this specific URL path isn't just documentation—it's a live link to your code.

Now that the gateway "map" is correct, are you ready to learn how to **secure** this path using an **API Key**?

## Fix the Test Event Data for Cloud Function

You have a working Cloud Function that calculates tax and totals for a given amount, but the current test event file (event.json) contains the wrong data structure. The function expects a specific JSON payload, but the file has incorrect fields that will cause errors or unexpected results.

Your task is to:

    Read the Cloud Function handler code to understand which data structure it expects
    Fix the event.json file with the correct JSON payload
    Test your changes locally using the Functions Framework to verify that everything works

This exercise will help you debug event data issues and create proper test files for your Cloud Functions before deployment.

```json
{
  "value": 150,
  "currency": "USD"
}

```

```python
import json

def calculate_tax(request):
    try:
        body = request.get_json(silent=True) or {}
        amount = float(body.get("amount", 0))
        tax = round(amount * 0.07, 2)
        total = round(amount + tax, 2)
        response = {
            "amount": amount,
            "tax": tax,
            "total": total
        }
        return (json.dumps(response), 200, {"Content-Type": "application/json"})
    except Exception as e:
        return (json.dumps({"error": str(e)}), 400, {"Content-Type": "application/json"})

```

To fix the `event.json` file, we need to look at how the Python function retrieves data. Inside the code, the line `amount = float(body.get("amount", 0))` tells us that the function is looking for a key specifically named **"amount"**.

Currently, your JSON file uses the key `"value"`. Because of the way `.get("amount", 0)` works, the function isn't crashing, but it is defaulting the amount to `0`, which results in a tax of `0` and a total of `0`.

### Fixed `event.json`

Update your file to match the expected schema:

```json
{
  "amount": 150
}
```

### Why this fix works

1.  **Key Matching:** The dictionary key in your JSON (`"amount"`) now exactly matches the string inside `body.get("amount", 0)`.
2.  **Data Casting:** The value `150` is a valid number. The function will successfully run `float(150)`, allowing the math logic to proceed.
3.  **Simplicity:** While your original JSON had a `"currency"` field, the current function doesn't use it. Including extra data won't break this specific function, but keeping your test events "lean" makes debugging much easier.

### Testing your fix
Once you've saved the file, you can test it locally with the Functions Framework using this command in your terminal:

```bash
curl -X POST http://localhost:8080 \
   -H "Content-Type: application/json" \
   -d @event.json
```

**Expected Result:**
```json
{"amount": 150.0, "tax": 10.5, "total": 160.5}
```

This confirms that your "Event" (the input) and your "Handler" (the code) are perfectly aligned. Ready to see how to handle cases where the input might be completely missing?

## Complete the OpenAPI Specification

You've learned how to define Cloud Functions and connect them to API Gateway using configuration files. Now, let's focus on the foundation of every GCP serverless deployment: properly defining your API and Cloud Function connection in an OpenAPI specification.

You have a working Python handler that calculates tax and a complete test event file, but the openapi.yaml file is missing some key properties that tell API Gateway how to route requests to your Cloud Function. Your job is to complete the openapi.yaml file by adding the missing configuration.

Look at the main.py file to understand:

    What the handler function is called
    What the endpoint should accept and return

Then, fill in the missing properties in the OpenAPI spec so API Gateway knows where to send requests, what the request and response should look like, and how to connect to your deployed Cloud Function.

This exercise will give you hands-on experience with the core OpenAPI properties that every GCP API Gateway config needs.


```yaml
openapi: 3.0.0
info:
  title: Tax Calculator API
  version: 1.0.0
paths:
  /calculate:
    post:
      summary: Calculate tax and total
      operationId: calculateTax
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              # TODO: Add the "amount" property to the request schema
      responses:
        '200':
          description: Calculation result
          content:
            application/json:
              schema:
                type: object
                # TODO: Add "amount", "tax", and "total" properties to the response schema
      # TODO: Add x-google-backend to connect this endpoint to your Cloud Function

```

To complete the `openapi.yaml` file, you need to define the data "contracts" (the schema) and the specific Google extension that bridges the gap between the Gateway and your actual code. 

Think of the **schema** as a set of rules that tells the API Gateway what data to expect, and the **`x-google-backend`** as the physical address of the server where your Python code is waiting to run.



### Completed `openapi.yaml`

```yaml
openapi: 3.0.0
info:
  title: Tax Calculator API
  version: 1.0.0
paths:
  /calculate:
    post:
      summary: Calculate tax and total
      operationId: calculateTax
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              # TODO: Add the "amount" property to the request schema
              properties:
                amount:
                  type: number
      responses:
        '200':
          description: Calculation result
          content:
            application/json:
              schema:
                type: object
                # TODO: Add "amount", "tax", and "total" properties to the response schema
                properties:
                  amount:
                    type: number
                  tax:
                    type: number
                  total:
                    type: number
      # TODO: Add x-google-backend to connect this endpoint to your Cloud Function
      x-google-backend:
        address: https://REGION-PROJECT_ID.cloudfunctions.net/calculate_tax
```

### Why these sections are vital:

* **Request Schema:** By defining `amount` as a `number`, the API Gateway can actually validate incoming requests before they even reach your Python code. If someone sends text instead of a number, the Gateway can reject it immediately, saving you execution costs.
* **Response Schema:** This acts as documentation. It tells developers using your API exactly what keys they will see in the result (`amount`, `tax`, and `total`).
* **`x-google-backend`**: This is the most critical part of the GCP configuration. Without this specific "vendor extension," the API Gateway is just a document; with it, it becomes a functional router that knows to send the data to your specific Cloud Function URL.

### Pro-Tip: The Address Match
The `address` in `x-google-backend` must end with the name of the function as it appears in the GCP Console. If your function is named `calculate_tax` in the deploy command, ensure the URL matches exactly, or you'll get a `502 Bad Gateway` error!

You've now successfully configured the "brain" of your API. Ready to learn how to **restrict access** so only people with a valid API Key can use this calculator?

## Build and Configure a Complete GCP Serverless Tax Calculator API

Now that you've learned how to define, test, and deploy serverless APIs on GCP using Cloud Functions and API Gateway, it's time to put everything together and build a complete serverless application from scratch.

You are provided with a working Cloud Function in main.py that calculates tax and total amounts. Your job is to create all the necessary configuration and test files to make this function work as a deployed API on GCP.

Your tasks are:

    Write a complete openapi.yaml file that defines a POST endpoint /calculate and connects it to your Cloud Function.
    Create a sample event.json file with test data for local testing.
    Make sure your configuration matches what the function expects as input and output.

Read the main.py file to understand the expected request and response structure. This exercise will show you can build and configure a GCP serverless API from the ground up using the tools and formats you've learned.

```yaml
# TODO: Set openapi version to 3.0.0
# TODO: Add info section with title "Tax Calculator API" and version "1.0.0"
# TODO: Define a /calculate POST endpoint
# TODO: Add a summary and operationId for the endpoint
# TODO: Specify requestBody as required, with application/json content and an "amount" property of type number
# TODO: Define a 200 response with application/json content, including "amount", "tax", and "total" properties (all numbers)
# TODO: Use x-google-backend to connect this endpoint to your Cloud Function URL (replace REGION and PROJECT_ID as needed)

```

```json
# TODO: Create a JSON object with an "amount" field for testing, e.g. {"amount": 150}
```

```python
import json

def calculate_tax(request):
    try:
        body = request.get_json(silent=True) or {}
        amount = float(body.get("amount", 0))
        tax = round(amount * 0.07, 2)
        total = round(amount + tax, 2)
        response = {
            "amount": amount,
            "tax": tax,
            "total": total
        }
        return (json.dumps(response), 200, {"Content-Type": "application/json"})
    except Exception as e:
        return (json.dumps({"error": str(e)}), 400, {"Content-Type": "application/json"})
```

Building a serverless API on GCP requires a tight handshake between your code, your API definition, and your cloud infrastructure. Based on your `main.py` logic, here is the complete configuration to get your Tax Calculator up and running.

---

## 1. API Configuration: `openapi.yaml`

This file acts as the contract for your API. It defines the `/calculate` endpoint and uses the `x-google-backend` extension to bridge the API Gateway to your specific Cloud Function.

```yaml
openapi: 3.0.0
info:
  title: Tax Calculator API
  description: A serverless API to calculate tax and totals.
  version: 1.0.0

paths:
  /calculate:
    post:
      summary: Calculate tax and total for a given amount
      operationId: calculateTax
      x-google-backend:
        # Replace PROJECT_ID and REGION with your actual GCP details
        address: https://REGION-PROJECT_ID.cloudfunctions.net/calculate_tax
      requestBody:
        description: The amount to calculate tax for
        required: true
        content:
          application/json:
            schema:
              type: object
              required:
                - amount
              properties:
                amount:
                  type: number
                  example: 150.00
      responses:
        '200':
          description: Calculation successful
          content:
            application/json:
              schema:
                type: object
                properties:
                  amount:
                    type: number
                  tax:
                    type: number
                  total:
                    type: number
        '400':
          description: Invalid input or calculation error
          content:
            application/json:
              schema:
                type: object
                properties:
                  error:
                    type: string
```

---

## 2. Local Test Data: `event.json`

This file mimics the body of an HTTP request. You can use it with the Functions Framework to test your `main.py` locally before deploying.

```json
{
  "amount": 150
}
```

---

## 3. Implementation Checklist

To ensure your deployment is successful, keep these three points in mind:

* **The Backend URL:** In the `openapi.yaml`, the `address` under `x-google-backend` must match the **Trigger URL** provided by GCP after you deploy your Cloud Function.
* **CORS & Security:** By default, API Gateway might require authentication. If you want this to be a public API, ensure the Cloud Function has the `allUsers` member with the `roles/cloudfunctions.invoker` role.
* **Local Testing:** You can test your function locally using the following command:
    ```bash
    functions-framework --target calculate_tax --debug
    ```
    Then, in a separate terminal, send the test event:
    ```bash
    curl -X POST http://localhost:8080 -H "Content-Type: application/json" -d @event.json
    ```

> **Note:** In your Python code, the line `amount = float(body.get("amount", 0))` ensures that even if a user sends an integer, the calculator processes it correctly as a floating-point number. This matches the `type: number` definition in your YAML perfectly.